# SP500 Executive Role Classification — Fine-Tune gpt-oss-20b

Based on the [official Unsloth gpt-oss notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/gpt-oss-(20B)-Fine-tuning.ipynb), adapted for SP500 executive classification data.

**Runtime**: A100 80GB recommended. Go to Runtime → Change runtime type → A100.

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    full_finetuning = False,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 128,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

## Load & Format Training Data

In [ ]:
# Clone repo with training data
REPO_DIR = "/content/finetune_sp500"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/daalbert981/finetune_sp500.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

In [ ]:
import json
from datasets import Dataset

DATA_PATH = os.path.join(REPO_DIR, "Training Datasets", "full_data_n_4977.jsonl")
with open(DATA_PATH, "r") as f:
    examples = [json.loads(line) for line in f]

print(f"Loaded {len(examples)} examples")

# Shuffle and split 95/5
import random
random.seed(42)
random.shuffle(examples)
split_idx = int(len(examples) * 0.95)

train_dataset = Dataset.from_list(examples[:split_idx])
valid_dataset = Dataset.from_list(examples[split_idx:])
print(f"Train: {len(train_dataset)}, Valid: {len(valid_dataset)}")

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

# Our data is already in OpenAI format (role/content) — no need for standardize_sharegpt
train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
valid_dataset = valid_dataset.map(formatting_prompts_func, batched = True)

print(f"Train samples: {len(train_dataset)}, Valid samples: {len(valid_dataset)}")
# Show FULL formatted text of first example to see exact tags
print("=== FULL FORMATTED TEXT ===")
print(train_dataset[0]["text"])

## Train

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 2,
        num_train_epochs = 3,
        warmup_ratio = 0.03,
        learning_rate = 2e-4,
        logging_steps = 10,
        eval_strategy = "no",
        save_strategy = "steps",
        save_steps = 200,
        save_total_limit = 2,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
        bf16 = True,
        max_seq_length = 2048,
        dataloader_num_workers = 4,
        dataloader_pin_memory = True,
    ),
)

print(f"Effective batch size: {8 * 2}")
print(f"Train samples: {len(train_dataset)}")
print(f"Approx training steps: ~{len(train_dataset) * 3 // 16}")

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")

## Quick Inference Test

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": train_dataset[0]["messages"][0]["content"]},
    {"role": "user", "content": "In 2020 the company 'apple inc' had an executive with the name jane doe, whose official role title was: 'senior vice president, human resources'."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 200, streamer = TextStreamer(tokenizer))

## Save to HuggingFace

In [ ]:
# Load HF token from Colab secrets
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

# Save locally
model.save_pretrained("sp500_exec_lora")
tokenizer.save_pretrained("sp500_exec_lora")
print("LoRA adapters saved locally.")

In [ ]:
# Push to HuggingFace — use mxfp4 (correct format for gpt-oss MoE)
model.push_to_hub_merged(
    "daresearch/sp500-exec-classifier",
    tokenizer,
    save_method = "mxfp4",
    token = HF_TOKEN,
)
print("Model pushed to https://huggingface.co/daresearch/sp500-exec-classifier")

# Also push LoRA adapters separately (small, flexible)
model.push_to_hub("daresearch/sp500-exec-classifier-lora", token = HF_TOKEN)
tokenizer.push_to_hub("daresearch/sp500-exec-classifier-lora", token = HF_TOKEN)
print("LoRA adapters pushed to https://huggingface.co/daresearch/sp500-exec-classifier-lora")